In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, lr_init=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="greedy")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.33209681510925293
epoch:  1, loss: 0.29586061835289
epoch:  2, loss: 0.252820760011673
epoch:  3, loss: 0.20426543056964874
epoch:  4, loss: 0.16822898387908936
epoch:  5, loss: 0.044633615761995316
epoch:  6, loss: 0.034813668578863144
epoch:  7, loss: 0.015580957755446434
epoch:  8, loss: 0.010920948348939419
epoch:  9, loss: 0.009096190333366394
epoch:  10, loss: 0.007651616353541613
epoch:  11, loss: 0.006394044030457735
epoch:  12, loss: 0.00552243459969759
epoch:  13, loss: 0.00491629634052515
epoch:  14, loss: 0.004429695662111044
epoch:  15, loss: 0.00403568334877491
epoch:  16, loss: 0.0037700794637203217
epoch:  17, loss: 0.0034761407878249884
epoch:  18, loss: 0.0032601358834654093
epoch:  19, loss: 0.0030475931707769632
epoch:  20, loss: 0.0028815974947065115
epoch:  21, loss: 0.0027472611982375383
epoch:  22, loss: 0.0026661145966500044
epoch:  23, loss: 0.002650537993758917
epoch:  24, loss: 0.0025378752034157515
epoch:  25, loss: 0.002470427891239524
e

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9811785221099854
Test metrics:  R2 = 0.6491892337799072


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, lr_init=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.5275135040283203
epoch:  1, loss: 0.47887101769447327
epoch:  2, loss: 0.3122180104255676
epoch:  3, loss: 0.3007291853427887
epoch:  4, loss: 0.09317389875650406
epoch:  5, loss: 0.07131300866603851
epoch:  6, loss: 0.0249253548681736
epoch:  7, loss: 0.012706771492958069
epoch:  8, loss: 0.00899804849177599
epoch:  9, loss: 0.007196490652859211
epoch:  10, loss: 0.006182669196277857
epoch:  11, loss: 0.005562931764870882
epoch:  12, loss: 0.005149688106030226
epoch:  13, loss: 0.004795355256646872
epoch:  14, loss: 0.004533565603196621
epoch:  15, loss: 0.004366121720522642
epoch:  16, loss: 0.004240436013787985
epoch:  17, loss: 0.004099784418940544
epoch:  18, loss: 0.003980954643338919
epoch:  19, loss: 0.003874026471748948
epoch:  20, loss: 0.0037642617244273424
epoch:  21, loss: 0.0036586157511919737
epoch:  22, loss: 0.0035688113421201706
epoch:  23, loss: 0.003504839725792408
epoch:  24, loss: 0.003432626137509942
epoch:  25, loss: 0.003369845449924469
epoch

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.940197765827179
Test metrics:  R2 = 0.8073347806930542


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, lr_init=1, c1=1e-4, c2=0.9, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.3256497085094452
epoch:  1, loss: 0.31945887207984924
epoch:  2, loss: 0.3131825625896454
epoch:  3, loss: 0.30699077248573303
epoch:  4, loss: 0.3009214699268341
epoch:  5, loss: 0.2950107455253601
epoch:  6, loss: 0.2891528308391571
epoch:  7, loss: 0.28371796011924744
epoch:  8, loss: 0.27855804562568665
epoch:  9, loss: 0.2738342583179474
epoch:  10, loss: 0.26934149861335754
epoch:  11, loss: 0.2649751901626587
epoch:  12, loss: 0.26062479615211487
epoch:  13, loss: 0.25615012645721436
epoch:  14, loss: 0.25174638628959656
epoch:  15, loss: 0.2513331472873688
epoch:  16, loss: 0.2509305775165558
epoch:  17, loss: 0.24706673622131348
epoch:  18, loss: 0.24670594930648804
epoch:  19, loss: 0.2463480532169342
epoch:  20, loss: 0.2429661899805069
epoch:  21, loss: 0.23969928920269012
epoch:  22, loss: 0.23937654495239258
epoch:  23, loss: 0.23905859887599945
epoch:  24, loss: 0.23602080345153809
epoch:  25, loss: 0.23569682240486145
epoch:  26, loss: 0.2353851646184

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9314315319061279
Test metrics:  R2 = 0.8256317973136902


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, lr_init=1, c1=1e-4, c2=0.9, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.24216251075267792
epoch:  1, loss: 0.216054305434227
epoch:  2, loss: 0.17993061244487762
epoch:  3, loss: 0.17993061244487762
epoch:  4, loss: 0.17993061244487762
epoch:  5, loss: 0.17993061244487762
epoch:  6, loss: 0.17993061244487762
epoch:  7, loss: 0.17993061244487762
epoch:  8, loss: 0.17993061244487762
epoch:  9, loss: 0.17993061244487762
epoch:  10, loss: 0.17993061244487762
epoch:  11, loss: 0.17993061244487762
epoch:  12, loss: 0.17993061244487762
epoch:  13, loss: 0.17993061244487762
epoch:  14, loss: 0.17993061244487762
epoch:  15, loss: 0.17993061244487762
epoch:  16, loss: 0.17993061244487762
epoch:  17, loss: 0.17993061244487762
epoch:  18, loss: 0.17993061244487762
epoch:  19, loss: 0.17993061244487762
epoch:  20, loss: 0.17993061244487762
epoch:  21, loss: 0.17993061244487762
epoch:  22, loss: 0.17993061244487762
epoch:  23, loss: 0.17993061244487762
epoch:  24, loss: 0.17993061244487762
epoch:  25, loss: 0.17993061244487762
epoch:  26, loss: 0.1799

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -630.7691040039062
Test metrics:  R2 = -566.5843505859375


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, lr_init=1, c1=1e-4, c2=0.9, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.1762305051088333
epoch:  1, loss: 0.1762305051088333
epoch:  2, loss: 0.1762305051088333
epoch:  3, loss: 0.1762305051088333
epoch:  4, loss: 0.1762305051088333
epoch:  5, loss: 0.1762305051088333
epoch:  6, loss: 0.1762305051088333
epoch:  7, loss: 0.1762305051088333
epoch:  8, loss: 0.1762305051088333
epoch:  9, loss: 0.1762305051088333
epoch:  10, loss: 0.1762305051088333
epoch:  11, loss: 0.1762305051088333
epoch:  12, loss: 0.1762305051088333
epoch:  13, loss: 0.1762305051088333
epoch:  14, loss: 0.1762305051088333
epoch:  15, loss: 0.1762305051088333
epoch:  16, loss: 0.1762305051088333
epoch:  17, loss: 0.1762305051088333
epoch:  18, loss: 0.1762305051088333
epoch:  19, loss: 0.1762305051088333
epoch:  20, loss: 0.1762305051088333
epoch:  21, loss: 0.1762305051088333
epoch:  22, loss: 0.1762305051088333
epoch:  23, loss: 0.1762305051088333
epoch:  24, loss: 0.1762305051088333
epoch:  25, loss: 0.1762305051088333
epoch:  26, loss: 0.1762305051088333
epoch:  27,

In [14]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -26538.5
Test metrics:  R2 = -22900.728515625
